In [34]:
import torch
import torchvision.models as models
import timm
import backbone.Custom as Custom
from torch.utils.data import Dataset, DataLoader
import importlib
importlib.reload(Custom)


<module 'backbone.Custom' from '/users/koketso/Feature_extraction/spectra_for_features/backbone/Custom.py'>

# Data

In [21]:
def Galaxy_zoo_data_loaders(galaxyzoo_dir = "/idia/projects/camil/Koketso/galaxyzoo2",
                            galaxyzooq_dir = "/idia/projects/camil/Koketso/galaxyzoo/resized/galaxy_zoo_class_new",
                              valsplit = 0.05,
                            train_split = 0.8,
                            num_workers = 30,
                            batch_size = 128,
                            resize = 224,
                            crop_size = 224):

    dataset = Custom.dataset(galaxyzoo_dir)
    names = [name[0].split('/')[-1] for name in dataset.imgs]

    #classification validation

    classification_val_dataset = Custom.dataset(galaxyzooq_dir)

    datasets = Custom.train_val_dataset(dataset, 
                                        val_split = valsplit
                                        ,train_size = train_split)

    #Traning

    transformed_train_dataset = Custom.Custom(datasets['train'],
                                            names = names,
                                            resize = resize,
                                           crop = crop_size,
                                           )


    loader = DataLoader(transformed_train_dataset, 
                            batch_size, 
                            shuffle = True,
                            num_workers = num_workers)

    #validation

    transformed_val_dataset = Custom.Custom(datasets['val'],
                                            names = names,
                                            resize = resize,
                                           crop = crop_size,
                                           )

    val_loader = DataLoader(transformed_val_dataset, 
                            batch_size, 
                            shuffle = True,
                            num_workers = num_workers)


    #Classification validation

    transformed_classification_val_dataset = Custom.Custom_labelled(classification_val_dataset,
                                            names = names,
                                            resize = resize,
                                           crop = crop_size,
                                           )



    class_loader = DataLoader(transformed_classification_val_dataset, 
                            batch_size, 
                            shuffle = True,
                            num_workers = num_workers)

    return loader, val_loader, class_loader

In [23]:
train,val,class_  = Galaxy_zoo_data_loaders()


# Model

In [24]:
encoder = timm.create_model('hf_hub:mwalmsley/zoobot-encoder-convnext_base', pretrained=True, num_classes=0)
encoder

ConvNeXt(
  (stem): Sequential(
    (0): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((128,), eps=1e-06, elementwise_affine=True)
  )
  (stages): Sequential(
    (0): ConvNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvNeXtBlock(
          (conv_dw): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=128)
          (norm): LayerNorm((128,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=128, out_features=512, bias=True)
            (act): GELU()
            (drop1): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (fc2): Linear(in_features=512, out_features=128, bias=True)
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (shortcut): Identity()
          (drop_path): Identity()
        )
        (1): ConvNeXtBlock(
          (conv_dw): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), g

In [ ]:
Custom.get_representations(model  = encoder,
                           loader = val,
                           encoder = True)

In [11]:
weights = torch.load("pytorch_model.bin")
models.convnext_base(weights= None)

ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((128,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=128)
          (1): Permute()
          (2): LayerNorm((128,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=128, out_features=512, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=512, out_features=128, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=128)
          (1): Permute()
          (2): LayerNorm((128,), eps=1e-06, elementwise_affine=True)
          (3): Linear(

In [10]:
zoobot = models.convnext_base(weights= None)
zoobot.load_state_dict(weights)

RuntimeError: Error(s) in loading state_dict for ConvNeXt:
	Missing key(s) in state_dict: "features.0.0.weight", "features.0.0.bias", "features.0.1.weight", "features.0.1.bias", "features.1.0.layer_scale", "features.1.0.block.0.weight", "features.1.0.block.0.bias", "features.1.0.block.2.weight", "features.1.0.block.2.bias", "features.1.0.block.3.weight", "features.1.0.block.3.bias", "features.1.0.block.5.weight", "features.1.0.block.5.bias", "features.1.1.layer_scale", "features.1.1.block.0.weight", "features.1.1.block.0.bias", "features.1.1.block.2.weight", "features.1.1.block.2.bias", "features.1.1.block.3.weight", "features.1.1.block.3.bias", "features.1.1.block.5.weight", "features.1.1.block.5.bias", "features.1.2.layer_scale", "features.1.2.block.0.weight", "features.1.2.block.0.bias", "features.1.2.block.2.weight", "features.1.2.block.2.bias", "features.1.2.block.3.weight", "features.1.2.block.3.bias", "features.1.2.block.5.weight", "features.1.2.block.5.bias", "features.2.0.weig